# Who judges the judge? Checking an LLM judge against human experts

Many teams let a strong LLM grade the outputs of their AI products. Before trusting it, I check three
things on public data:

1. Does the judge (GPT-4) agree with human experts as often as experts agree with each other?
2. Is that true for every type of task?
3. Does the judge reward longer answers more than experts do?

**Data:** [MT-Bench human judgments](https://huggingface.co/datasets/lmsys/mt_bench_human_judgments)
(LMSYS, CC BY 4.0): expert and GPT-4 votes on pairs of answers to 80 questions in 8 categories.
Paper: Zheng et al., *Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena*, NeurIPS 2023.

Runs in a few seconds, no GPU needed: *Runtime → Run all*.

## 1. Download the data (pinned versions)

In [ ]:
import json
import urllib.request
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score

DATA, RESULTS, FIGURES = Path("data"), Path("results"), Path("figures")
for folder in (DATA, RESULTS, FIGURES):
    folder.mkdir(exist_ok=True)

HF = "https://huggingface.co/datasets/lmsys/mt_bench_human_judgments/resolve/f7d2896d2cc5d80f8b55c2bbc722613555233c25/data/"
FILES = {
    "human.parquet": HF + "human-00000-of-00001-25f4910818759289.parquet",
    "gpt4_pair.parquet": HF + "gpt4_pair-00000-of-00001-c0b431264a82ddc0.parquet",
    # questions and their category, from the FastChat repository (Apache-2.0)
    "question.jsonl": "https://raw.githubusercontent.com/lm-sys/FastChat/"
                      "b494d0c6b4e7935f1764f8439e75da3e66beccc7/fastchat/llm_judge/data/mt_bench/question.jsonl",
}
for name, url in FILES.items():
    if not (DATA / name).exists():
        req = urllib.request.Request(url, headers={"User-Agent": "who-judges-the-judge"})
        with urllib.request.urlopen(req, timeout=60) as r:
            (DATA / name).write_bytes(r.read())
print(sorted(p.name for p in DATA.iterdir()))

## 2. Load and normalise the votes

Each vote says which of two answers is better: `a`, `b` or `tie`.

- "tie" and "tie (bothbad)" both mean *no preference*, so they are merged.
- The two models of each comparison are put in **alphabetical order** (and the vote is swapped with
  them). Otherwise the same opinion seen as (GPT-4 vs Claude) and (Claude vs GPT-4) would look like a
  disagreement.
- An *item* is one question, one turn and one pair of models: all votes on the same item are compared.

In [ ]:
questions = {}
for line in (DATA / "question.jsonl").read_text(encoding="utf-8").splitlines():
    q = json.loads(line)
    questions[q["question_id"]] = q


def to_vote(winner):
    w = str(winner).lower()
    if "tie" in w:
        return "tie"
    return {"model_a": "a", "model_b": "b"}[w]


def answer(conversation, turn):
    assistant = [m["content"] for m in conversation if m["role"] == "assistant"]
    return assistant[turn - 1] if len(assistant) >= turn else ""


rows = []
for file, source in [("human.parquet", "human"), ("gpt4_pair.parquet", "gpt4")]:
    for r in pd.read_parquet(DATA / file).itertuples(index=False):
        vote = to_vote(r.winner)
        m1, m2 = r.model_a, r.model_b
        ans1, ans2 = answer(r.conversation_a, r.turn), answer(r.conversation_b, r.turn)
        if m1 > m2:  # alphabetical order, vote swapped accordingly
            m1, m2, ans1, ans2 = m2, m1, ans2, ans1
            vote = {"a": "b", "b": "a", "tie": "tie"}[vote]
        q = questions[r.question_id]
        rows.append(dict(source=source, judge=r.judge, category=q["category"], turn=r.turn,
                         model_1=m1, model_2=m2, vote=vote, len_1=len(ans1), len_2=len(ans2),
                         question=q["turns"][r.turn - 1], answer_1=ans1, answer_2=ans2,
                         key=f"{r.question_id}|{r.turn}|{m1}|{m2}"))
votes = pd.DataFrame(rows)

hum, gpt = votes[votes.source == "human"], votes[votes.source == "gpt4"]
print(f"{len(hum)} expert votes ({hum.judge.nunique()} experts), {len(gpt)} GPT-4 votes")
votes.head(3)

## 3. Pairs of votes on the same item

- **expert vs expert** (the *baseline*): every pair of two different experts who voted the same item.
  A judge cannot be expected to agree with humans more than humans agree with each other.
- **GPT-4 vs expert**: the GPT-4 vote paired with each expert vote on that item.

In [ ]:
gpt_by_key = gpt.groupby("key").vote.apply(list)
expert_pairs, judge_pairs = [], []
for key, g in hum.groupby("key"):
    cat = g.category.iloc[0]
    for (_, x), (_, y) in combinations(g.iterrows(), 2):
        if x.judge != y.judge:
            expert_pairs.append(dict(category=cat, v1=x.vote, v2=y.vote))
    for gv in gpt_by_key.get(key, []):
        for hv in g.vote:
            judge_pairs.append(dict(category=cat, v1=gv, v2=hv))
expert_pairs, judge_pairs = pd.DataFrame(expert_pairs), pd.DataFrame(judge_pairs)


def no_ties(p):
    """Keep pairs where both sides picked a winner."""
    return p[(p.v1 != "tie") & (p.v2 != "tie")]


def agreement(p):
    return (p.v1 == p.v2).mean()

## 4. Overall agreement

Reported with ties excluded and included, plus **Cohen's kappa** (agreement corrected for chance:
0 = like flipping a coin, 1 = perfect).

In [ ]:
overall = pd.DataFrame([
    dict(pair=name, n_pairs=len(p),
         agreement_ties_excluded=agreement(no_ties(p)),
         agreement_with_ties=agreement(p),
         cohen_kappa=cohen_kappa_score(p.v1, p.v2))
    for name, p in [("expert vs expert", expert_pairs), ("GPT-4 vs expert", judge_pairs)]
]).round(3)
overall.to_csv(RESULTS / "agreement_overall.csv", index=False)
overall

## 5. Agreement by task type (ties excluded)

In [ ]:
bycat = []
for c in sorted(votes.category.unique()):
    j, e = no_ties(judge_pairs[judge_pairs.category == c]), no_ties(expert_pairs[expert_pairs.category == c])
    bycat.append(dict(category=c, gpt4_vs_expert=agreement(j), n_gpt4_pairs=len(j),
                      expert_vs_expert=agreement(e), n_expert_pairs=len(e)))
bycat = pd.DataFrame(bycat)
bycat["gap"] = bycat.gpt4_vs_expert - bycat.expert_vs_expert
bycat = bycat.sort_values("gpt4_vs_expert").round(3)
bycat.to_csv(RESULTS / "agreement_by_category.csv", index=False)

fig, ax = plt.subplots(figsize=(7.5, 4.2))
y = np.arange(len(bycat))
ax.barh(y, bycat.gpt4_vs_expert, color="#1f6f8b", label="GPT-4 vs expert")
ax.scatter(bycat.expert_vs_expert, y, marker="|", s=300, color="#e07a1f", zorder=3, label="expert vs expert")
ax.set_yticks(y, bycat.category)
ax.set_xlim(0.5, 1.0)
ax.set_xlabel("agreement on votes with a winner (ties excluded)")
ax.set_title("LLM judge vs human experts, by task type")
ax.legend(loc="lower right", fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES / "agreement_by_category.png", dpi=150)
plt.show()
bycat

## 6. Does the judge prefer longer answers?

Only items where one answer is at least 20% longer (in characters), and only items voted by both GPT-4
and experts, so both are compared on exactly the same answers.

In [ ]:
v = votes[votes.vote != "tie"].copy()
v["ratio"] = v[["len_1", "len_2"]].max(axis=1) / v[["len_1", "len_2"]].min(axis=1).clip(lower=1)
v = v[v.ratio >= 1.2]
both = set(v[v.source == "human"].key) & set(v[v.source == "gpt4"].key)
v = v[v.key.isin(both)]
v["picked_longer"] = np.where(v.len_1 > v.len_2, v.vote == "a", v.vote == "b")
v["length_gap"] = pd.cut(v.ratio, [1.2, 2, np.inf], right=False, labels=["1.2-2x longer", ">2x longer"])
lp = v.groupby(["length_gap", "source"], observed=True).picked_longer.agg(share="mean", n="size").reset_index()
lp.round(3).to_csv(RESULTS / "length_preference.csv", index=False)

table = lp.pivot(index="length_gap", columns="source", values="share")
ax = table[["human", "gpt4"]].rename(columns={"human": "experts", "gpt4": "GPT-4 judge"}).plot.bar(
    rot=0, color=["#e07a1f", "#1f6f8b"], figsize=(6, 3.8))
ax.axhline(0.5, color="grey", lw=0.8, ls="--")
ax.set_ylim(0, 1)
ax.set_xlabel("")
ax.legend(fontsize=8)
ax.set_ylabel("share of votes for the longer answer")
ax.set_title("Does the judge reward length more than experts?")
plt.tight_layout()
plt.savefig(FIGURES / "length_preference.png", dpi=150)
plt.show()
lp.round(3)

## 7. Clear disagreements, to read by hand

Items where at least two experts all picked the same winner and GPT-4 picked the other answer.
They are saved in `results/disagreements.csv` with an empty `my_note` column.

In [ ]:
gpt_vote = gpt.groupby("key").vote.first()
unanimous, rows = 0, []
for key, g in hum.groupby("key"):
    if len(g) < 2 or g.vote.nunique() != 1 or g.vote.iloc[0] == "tie":
        continue
    unanimous += 1
    experts, judge = g.vote.iloc[0], gpt_vote.get(key)
    if judge in ("a", "b") and judge != experts:
        m = g.iloc[0]
        rows.append(dict(category=m.category, turn=m.turn, model_1=m.model_1, model_2=m.model_2,
                         n_experts=len(g), experts_chose=experts, gpt4_chose=judge,
                         gpt4_chose_longer=(judge == "a") == (m.len_1 > m.len_2),
                         question=m.question, answer_1=m.answer_1, answer_2=m.answer_2, my_note=""))
dis = pd.DataFrame(rows)
dis.to_csv(RESULTS / "disagreements.csv", index=False)

print(f"items where >=2 experts all picked the same winner: {unanimous}")
print(f"GPT-4 picked the other answer: {len(dis)} ({len(dis) / unanimous:.1%})")
print(f"in these cases GPT-4 picked the longer answer {dis.gpt4_chose_longer.mean():.0%} of the time")
dis.category.value_counts()

Read one case (change `i` to see the others):

In [ ]:
i = 0
case = dis.iloc[i]
print(f"[{case.category}] experts chose {case.experts_chose}, GPT-4 chose {case.gpt4_chose}\n")
print("QUESTION:\n", case.question, "\n")
print(f"ANSWER 1 ({case.model_1}):\n", case.answer_1[:1500], "\n")
print(f"ANSWER 2 ({case.model_2}):\n", case.answer_2[:1500])

## What I take from it

- Overall the GPT-4 judge agrees with experts about as often as experts agree with each other: this
  reproduces the main result of the MT-Bench paper.
- The average hides differences by task type: in *coding* the judge stays below the expert baseline,
  while in open-ended tasks experts themselves disagree more.
- The judge picks the longer answer slightly more often than experts on the same items.
- Clear mistakes are rare, and they are not explained by length.

An LLM judge is a reasonable first filter, but its agreement should be measured per task type against a
human baseline, with length checked as a possible bias, before it replaces human review.

**Limits:** GPT-4 and models from 2023; 150-350 vote pairs per category, so differences of a few points
are uncertain; I did not run a judge myself; length is measured in characters.